# Campus Voice — HTML接続版
①準備→セッション再起動→②起動して元音声を選択。その後は表示されたHTML画面で解析・試聴・編集・動画生成を操作します。ipywidgetsや公開トンネルは使いません。

別タブの外部サイトへ接続する方式ではありません。無料Colabのノートブック内でHTMLを表示します。実Colabでの通し操作は未検証です。終了前は作業全体を保存してください。

In [ ]:
#@title ① 準備（完了後にセッション再起動）
"""Install a mutually compatible Colab UI/model stack; stop on failure."""
import subprocess
import sys

REQUIREMENTS = [
    'faster-whisper==1.2.1', 'av==16.0.1',
    'sentence-transformers==6.3.0', 'transformers==5.23.0',
    'huggingface-hub>=1.32,<3', 'scikit-learn', 'ipywidgets==7.8.5',
]


def setup_colab():
    print('1/2: 必要なライブラリを準備しています。', flush=True)
    subprocess.run([sys.executable,'-m','pip','install','--upgrade','--quiet',*REQUIREMENTS],check=True)
    print('2/2: 動画・字幕用のツールを準備しています。', flush=True)
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq','ffmpeg','fonts-noto-cjk'],check=True)
    print('準備完了。ここで必ず「ランタイム → セッションを再起動」を選び、その後②だけ実行してください。①の再実行は不要です。', flush=True)

setup_colab()


In [ ]:
#@title ② 元音声を選びHTML画面を開く
"""Candidate ranking; optional embeddings are computed in Colab, never the editor."""
import math
import re


def summarize_excerpt(segments, max_chars=220):
    """Extract representative sentences; never invent claims or paraphrase ASR.

    Character bigrams work without Japanese tokenizers. Keep selected sentences
    in chronological order; the UI explicitly labels this as an extractive summary.
    """
    sentences = []
    for segment in segments:
        for sentence in re.split(r'(?<=[。！？!?])', segment['text']):
            sentence = sentence.strip()
            if sentence and sentence not in sentences:
                sentences.append(sentence)
    if not sentences:
        return ''
    def grams(text):
        text = re.sub(r'\s+', '', text)
        return set(text[i:i+2] for i in range(len(text)-1))
    features = [grams(t) for t in sentences]
    weights = {}
    for feature in features:
        for gram in feature:
            weights[gram] = weights.get(gram, 0) + 1
    ranking = sorted(range(len(sentences)), key=lambda i: sum(weights[g] for g in features[i])/max(1,len(features[i])) + min(len(sentences[i]),80)/160, reverse=True)
    selected = []
    for i in ranking:
        if any(len(features[i]&features[j])/max(1,len(features[i]|features[j]))>.8 for j in selected):
            continue
        selected.append(i)
        if len(selected)==2:
            break
    text = ' … '.join(sentences[i] for i in sorted(selected))
    return text if len(text)<=max_chars else text[:max_chars-1]+'…'


def suggest_candidates(segments, embeddings=None, limit=10):
    """Rank 30–60s windows with topic, tempo and transcript reaction cues.

    Embeddings must align with segments. Reaction words are not laughter detection.
    Short recordings return fewer candidates rather than duplicating intervals.
    """
    if embeddings is not None and len(embeddings) != len(segments):
        raise ValueError('字幕と埋め込みの件数が一致しません')
    if not segments:
        return []
    boundaries = set()
    if embeddings is not None:
        for i in range(1, len(segments)):
            a, b = embeddings[i - 1], embeddings[i]
            norm = math.sqrt(sum(x*x for x in a) * sum(x*x for x in b))
            similarity = sum(x*y for x, y in zip(a, b)) / norm if norm else 1
            if similarity < .45:
                boundaries.add(i)
    windows = []
    for i, first in enumerate(segments):
        # A long individual subtitle cannot make a 30–60s candidate safely.
        selected = []
        for j in range(i, len(segments)):
            if segments[j]['end'] - first['start'] > 60:
                break
            selected.append(segments[j])
            duration = segments[j]['end'] - first['start']
            if duration < 30:
                continue
            gap = segments[j+1]['start']-segments[j]['end'] if j+1 < len(segments) else 2
            clean_end = gap >= .8 or j+1 in boundaries or j == len(segments)-1
            text = ''.join(s['text'] for s in selected)
            speech = sum(s['end']-s['start'] for s in selected)
            density = min(1, speech / duration)
            tempo = min(1, len(selected) / duration * 5)
            reactions = len(re.findall(r'すごい|面白|おもしろ|まさか|なるほど|[!！]|[（(]笑[)）]', text))
            topic = i in boundaries or i == 0
            score = .4*density + .25*tempo + .15*min(1,reactions/3) + .1*topic + .1*clean_end
            reasons = [f'発話密度 {density:.0%}', f'発話区間 {len(selected)}件']
            if topic and embeddings is not None:
                reasons.append('話題の区切りから開始')
            if reactions:
                reasons.append(f'反応語 {reactions}件（文字情報）')
            if clean_end:
                reasons.append('区切りで終了')
            windows.append(dict(start=first['start'], end=segments[j]['end'],
                                title=first['text'][:40], reason=' / '.join(reasons),
                                score=round(score,4), density=round(density,3),
                                reactions=reactions, topic_start=bool(topic and embeddings is not None),
                                tempo=round(tempo,3), clean_end=clean_end, opening=topic))
    chosen = []
    ranked = sorted(windows, key=lambda c: c['score'], reverse=True)
    def compatible(candidate):
        return not any(max(0,min(candidate['end'], c['end'])-max(candidate['start'],c['start'])) /
                       min(candidate['end']-candidate['start'],c['end']-c['start']) > .25 for c in chosen)
    # Reserve one eligible candidate per time band before filling by quality.
    # Use actual timestamps, including long silences, rather than subtitle count.
    if ranked:
        first = segments[0]['start']
        span = max(1, segments[-1]['end']-first)
        bands = min(5, limit)
        for band in range(bands):
            pool = [c for c in ranked if min(bands-1,int(((c['start']+c['end'])/2-first)/span*bands)) == band]
            candidate = next((c for c in pool if compatible(c)), None)
            if candidate is not None:
                candidate['selection_basis'] = f'解析範囲の第{band+1}/{bands}時間帯から、得点上位の重複しない候補を確保'
                chosen.append(candidate)
    for candidate in ranked:
        if len(chosen) >= limit:
            break
        if compatible(candidate):
            candidate['selection_basis'] = '時間帯別の候補を確保した後、得点順で重複しない候補を追加'
            chosen.append(candidate)
    for candidate in chosen:
        selected = [s for s in segments if s['end']>candidate['start'] and s['start']<candidate['end']]
        candidate['summary'] = summarize_excerpt(selected)
        candidate['summary_method'] = 'extractive-v1'
        candidate['reasons'] = [
            f"発話密度 {candidate['density']:.0%}：無音が少ない区間を加点",
            f"発話テンポ指標 {candidate['tempo']:.0%}：字幕区間数／秒から計算（話者交代数ではありません）",
            candidate['selection_basis'],
        ]
        if candidate['topic_start']:
            candidate['reasons'].append('話題の開始位置と推定されたため加点（先頭または隣接字幕の意味の変化）')
        elif candidate['opening']:
            candidate['reasons'].append('解析範囲の先頭から始まるため加点（意味による判定ではありません）')
        if candidate['reactions']:
            candidate['reasons'].append(f"驚き・反応の文字表現 {candidate['reactions']}件を加点（音声の笑い検出ではありません）")
        if candidate['clean_end']:
            candidate['reasons'].append('無音・話題境界・解析末尾のいずれかで終わるため加点')
    return sorted(chosen, key=lambda c:c['score'], reverse=True)

"""Colab speaker clustering using a public WavLM speaker embedding model.

Labels are estimates per transcript segment, not overlapping-source separation.
"""

def labels_to_segments(segments, labels):
    if len(segments) != len(labels):
        raise ValueError('話者ラベルと字幕の件数が一致しません')
    names = {}
    result = []
    for segment, label in zip(segments, labels):
        if label not in names:
            names[label] = f'話者{len(names)+1}'
        result.append(dict(segment, speaker=names[label]))
    return result


def diarize_segments(audio_path, segments, num_speakers=2):
    """Cluster per-segment embeddings. Requires the expected speaker count.

    Uses at most 6 seconds of each utterance to bound model memory use. Short
    utterances and speaker changes within one subtitle are inherently uncertain.
    """
    import subprocess
    import tempfile
    from pathlib import Path
    import numpy as np
    import torch
    from transformers import AutoFeatureExtractor, AutoModelForAudioXVector
    from sklearn.cluster import AgglomerativeClustering

    if not isinstance(num_speakers, int) or not 1 <= num_speakers <= 10:
        raise ValueError('話者数は1〜10の整数にしてください')
    if len(segments) < num_speakers:
        raise ValueError('字幕数より話者数が多いため、話者数を減らしてください')
    if num_speakers == 1:
        return labels_to_segments(segments, [0]*len(segments))
    model_id = 'microsoft/wavlm-base-plus-sv'
    extractor = AutoFeatureExtractor.from_pretrained(model_id)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    speaker_model = AutoModelForAudioXVector.from_pretrained(model_id).to(device).eval()
    embeddings = []
    # Disk-backed PCM avoids loading an entire 50-minute waveform into RAM.
    with tempfile.TemporaryDirectory() as directory:
        pcm = Path(directory)/'voice.f32'
        subprocess.run(['ffmpeg','-y','-v','error','-i',str(audio_path),'-vn','-ar','16000','-ac','1','-f','f32le',str(pcm)],check=True)
        audio = np.memmap(pcm, dtype='float32', mode='r')
        try:
            for i, segment in enumerate(segments):
                start = int(segment['start']*16000)
                end = min(len(audio), int(segment['end']*16000), start+6*16000)
                if end <= start:
                    raise ValueError('字幕時刻が元音声の範囲を超えています')
                clip = np.array(audio[start:end])
                if len(clip) < 16000:
                    clip = np.pad(clip, (0,16000-len(clip)))
                inputs = extractor(clip, sampling_rate=16000, return_tensors='pt', padding=True)
                with torch.inference_mode():
                    vector = speaker_model(**{k:v.to(device) for k,v in inputs.items()}).embeddings
                    vector = torch.nn.functional.normalize(vector, dim=-1)
                embeddings.append(vector[0].cpu().numpy())
                if (i+1)%25 == 0:
                    print(f'話者解析: {i+1}/{len(segments)}字幕')
        finally:
            del audio
    labels = AgglomerativeClustering(n_clusters=num_speakers, metric='cosine', linkage='average').fit_predict(np.stack(embeddings))
    del speaker_model
    if device == 'cuda':
        torch.cuda.empty_cache()
    return labels_to_segments(segments, labels.tolist())

"""Validate pasted/file edits before replacing the current Colab edit."""
import json
import math


def parse_edit(text, audio_name):
    try:
        edit = json.loads(text)
    except (ValueError, TypeError) as error:
        raise ValueError('編集画面の「Colabへコピー」からコピーしたJSONを貼り付けてください') from error
    if not isinstance(edit, dict):
        raise ValueError('編集JSONが必要です')
    if edit.get('kind') == 'campus-voice-session':
        raise ValueError('作業全体のJSONではなく、編集画面の「Colabへコピー」を使ってください')
    def number(value):
        return isinstance(value, (float, int)) and not isinstance(value, bool) and math.isfinite(value)
    start, end = edit.get('start'), edit.get('end')
    if edit.get('version') != 1 or not number(start) or not number(end) or not 0 <= start < end or end-start > 600:
        raise ValueError('開始・終了を含む編集JSONが必要です（最大600秒）。解析JSONは使えません')
    if edit.get('audio_name') != audio_name:
        raise ValueError('解析時と同じ元音声を選択してください')
    segments = edit.get('segments')
    if not isinstance(segments, list) or not segments:
        raise ValueError('字幕が含まれていません')
    for segment in segments:
        if not isinstance(segment,dict) or not number(segment.get('start')) or not number(segment.get('end')) or not 0 <= segment['start'] < segment['end'] or not isinstance(segment.get('text'),str):
            raise ValueError('字幕の時刻・内容が不正です')
    volume = edit.get('bgm_volume', .12)
    if not number(volume) or not 0 <= volume <= 1:
        raise ValueError('BGM音量は0〜1にしてください')
    if edit.get('format') not in ('vertical','square','landscape'):
        raise ValueError('動画サイズが不正です')
    return edit

"""Shared FFmpeg renderer for the button-based Colab interface."""
import math
import subprocess
import tempfile
from pathlib import Path


def render_clip(audio_path, edit, bgm=None):
    audio_path = Path(audio_path).resolve()
    bgm = str(Path(bgm).resolve()) if bgm else None
    work = Path(tempfile.mkdtemp(prefix='campus-clip-'))
    start,end=float(edit['start']),float(edit['end']); duration=end-start
    assert math.isfinite(duration) and 0<duration<=600
    size={'vertical':(1080,1920),'square':(1080,1080),'landscape':(1920,1080)}[edit['format']]
    def stamp(t):
        cs=max(0,round(t*100)); h,cs=divmod(cs,360000); m,cs=divmod(cs,6000); s,cs=divmod(cs,100)
        return f'{h}:{m:02}:{s:02}.{cs:02}'
    def safe_text(t):
        return str(t).replace('\\','').replace('{','').replace('}','').replace('\n',r'\N').replace('\r','')
    w,h=size
    header=f"""[Script Info]
    ScriptType: v4.00+
    PlayResX: {w}
    PlayResY: {h}
    WrapStyle: 0
    [V4+ Styles]
    Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
    Style: Default,Noto Sans CJK JP,56,&H00FFFFFF,&H00FFFFFF,&H00203930,&H00203930,0,0,0,0,100,100,0,0,1,3,0,5,80,80,60,1
    [Events]
    Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
    """
    lines=[]
    for s in edit['segments']:
        a=max(start,float(s['start'])); b=min(end,float(s['end']))
        if b<=a: continue
        text=(str(s.get('speaker',''))+'：' if s.get('speaker') else '')+str(s['text'])
        lines.append(f'Dialogue: 0,{stamp(a-start)},{stamp(b-start)},Default,,0,0,0,,{safe_text(text)}')
    (work / 'captions.ass').write_text(header+'\n'.join(lines),encoding='utf-8')
    cmd=['ffmpeg','-y','-f','lavfi','-i',f'color=c=0x173e35:s={w}x{h}:r=30:d={duration}', '-ss',str(start),'-t',str(duration),'-i',str(audio_path)]
    if bgm:
        volume=float(edit.get('bgm_volume',0.12)); assert 0<=volume<=1
        cmd+=['-stream_loop','-1','-i',bgm,'-filter_complex',f'[2:a]volume={volume},afade=t=in:d=1,afade=t=out:st={max(0,duration-1)}:d=1[bg];[1:a][bg]amix=inputs=2:duration=first:normalize=0[a]','-map','0:v','-map','[a]']
    else: cmd+=['-map','0:v','-map','1:a']
    cmd+=['-vf','ass=captions.ass','-t',str(duration),'-c:v','libx264','-preset','fast','-threads','2','-pix_fmt','yuv420p','-c:a','aac','-movflags','+faststart','clip.mp4']
    subprocess.run(cmd,check=True,cwd=work,capture_output=True)
    return work / 'clip.mp4'

"""Analysis used by the Colab button UI. No persistent server or external API."""
import gc
import json
import subprocess
import tempfile
from pathlib import Path


def analyze_audio(audio_path, test_seconds=0, speakers=2, use_speakers=True, use_topics=True, count=10, report=print):
    from faster_whisper import WhisperModel
    import torch
    audio_path=Path(audio_path)
    work=Path(tempfile.mkdtemp(prefix='campus-analysis-'))
    source=audio_path
    if test_seconds:
        source=work/'trial.wav'
        subprocess.run(['ffmpeg','-y','-v','error','-i',str(audio_path),'-t',str(test_seconds),'-ar','16000','-ac','1',str(source)],check=True)
    gpu=torch.cuda.is_available()
    report('文字起こしを準備しています。初回はモデルをダウンロードします。')
    model=WhisperModel('small',device='cuda' if gpu else 'cpu',compute_type='float16' if gpu else 'int8')
    segments=[]
    try:
        stream,info=model.transcribe(str(source),language='ja',vad_filter=True,beam_size=5)
        for item in stream:
            if not item.text.strip():continue
            segments.append(dict(start=item.start,end=item.end,text=item.text.strip(),speaker=''))
            (work/'transcript-checkpoint.json').write_text(json.dumps(dict(audio_name=audio_path.name,segments=segments),ensure_ascii=False))
            if len(segments)%25==0:report(f'文字起こし: {item.end/60:.1f}分まで完了')
    except Exception:
        report(f'途中字幕を保存しました: {work / "transcript-checkpoint.json"}')
        raise
    finally:
        del model
        gc.collect()
        if gpu:torch.cuda.empty_cache()
    if not segments:raise ValueError('発話を検出できませんでした。元音声を確認してください。')
    speaker_status=dict(success=False,requested_speakers=speakers)
    if use_speakers:
        report('話者を推定しています。')
        try:
            segments=diarize_segments(source,segments,speakers)
            speaker_status.update(success=True,method='wavlm-utterance-clustering')
        except Exception as error:
            speaker_status['error']=str(error)
            report(f'話者推定を利用できませんでした。名前は手入力できます: {error}')
    embeddings=None
    if use_topics:
        report('話題の区切りを調べています。')
        try:
            from sentence_transformers import SentenceTransformer
            topic_model=SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',device='cpu')
            embeddings=topic_model.encode([s['text'] for s in segments],normalize_embeddings=True).tolist()
            del topic_model
        except Exception as error:
            report(f'話題モデルを利用できないため、他の指標で候補を選びます: {error}')
    candidates=suggest_candidates(segments,embeddings,count)
    project=dict(version=1,audio_name=audio_path.name,segments=segments,candidates=candidates,analysis=dict(test_seconds=test_seconds,speaker_diarization=speaker_status['success'],speaker_details=speaker_status,topic_analysis=embeddings is not None,laughter_detection=False,candidate_method='time-balanced-topic-tempo-v3'))
    report(f'完了: {len(segments)}字幕・{len(candidates)}候補。次の「候補と編集」へ進んでください。')
    return project

"""Notebook-local HTML callbacks. No HTTP server, tunnel or public endpoint."""
import base64
import json
import math
import subprocess
import tempfile
import threading
import uuid
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path


class HtmlBridge:
    def __init__(self,audio,bgm=None,analyze=analyze_audio,render=render_clip):
        self.audio=Path(audio);self.bgm=bgm;self.analyze=analyze;self.render=render
        self.pool=ThreadPoolExecutor(max_workers=1)
        self.lock=threading.Lock();self.job=None;self.project=None

    def call(self,action,data=None):
        data=data or {}
        try:
            with self.lock:
                return dict(ok=True,**self._call(action,data))
        except Exception as error:
            return dict(ok=False,error=str(error))

    def _call(self,action,data):
        if action=='hello':return dict(audio_name=self.audio.name,bgm=bool(self.bgm))
        if action in ('analyze','render','preview'):
            if self.job and not self.job['future'].done():raise ValueError('前の処理が実行中です。完了までお待ちください')
            if action=='analyze':
                seconds=data.get('seconds',0);speakers=data.get('speakers',2);count=data.get('count',10)
                if seconds not in (0,180) or type(speakers)!=int or not 1<=speakers<=10 or type(count)!=int or not 5<=count<=10:raise ValueError('解析設定が不正です')
                def task():
                    project=self.analyze(self.audio,test_seconds=seconds,speakers=speakers,count=count,use_speakers=bool(data.get('diarize',True)),use_topics=bool(data.get('topics',True)),report=report)
                    self.project=project
                    return dict(project=project)
            elif action=='render':
                edit=parse_edit(json.dumps(data.get('edit')),self.audio.name)
                def task():
                    duration=float(subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=nw=1:nk=1',str(self.audio)],capture_output=True,text=True,check=True).stdout)
                    if edit['end']>duration+.1:raise ValueError('編集範囲が元音声の長さを超えています')
                    report('動画を生成しています。')
                    return self.artifact(self.render(self.audio,edit,self.bgm if data.get('bgm',True) else None),'video/mp4')
            else:
                a,b=data.get('start'),data.get('end')
                if any(type(v) not in (int,float) or not math.isfinite(v) for v in (a,b)) or not 0<=a<b or b-a>620:raise ValueError('試聴範囲が不正です')
                def task():
                    report('試聴音声を準備しています。')
                    path=Path(tempfile.mkdtemp(prefix='campus-listen-'))/'preview.wav'
                    subprocess.run(['ffmpeg','-y','-v','error','-ss',str(a),'-i',str(self.audio),'-t',str(b-a),'-ar','16000','-ac','1',str(path)],check=True,capture_output=True)
                    return self.artifact(path,'audio/wav')
            messages=[]
            def report(message):
                messages.append(str(message));del messages[:-20]
            job_id=uuid.uuid4().hex
            self.job=dict(id=job_id,messages=messages,future=self.pool.submit(task))
            return dict(job=job_id)
        if action in ('status','chunk'):
            job=self.job
            if not job or data.get('job')!=job['id']:raise ValueError('処理が見つかりません。再接続・再実行してください')
            future=job['future']
            if action=='status':
                return dict(done=future.done(),messages=list(job['messages']),result=future.result() if future.done() else None)
            if not future.done():raise ValueError('まだ生成中です')
            result=future.result()
            if 'path' not in result:raise ValueError('ダウンロード対象がありません')
            offset=data.get('offset',0)
            if type(offset)!=int or offset<0 or offset>result['size']:raise ValueError('読み込み位置が不正です')
            with Path(result['path']).open('rb') as handle:
                handle.seek(offset);chunk=handle.read(512*1024)
            return dict(data=base64.b64encode(chunk).decode(),next=offset+len(chunk),done=offset+len(chunk)>=result['size'])
        raise ValueError('未知の操作です')

    @staticmethod
    def artifact(path,mime):
        path=Path(path)
        return dict(path=str(path),name=path.name,size=path.stat().st_size,mime=mime)


def launch_html_editor(page):
    from google.colab import files,output
    from IPython.display import display,HTML,JSON
    print('元音声1本と任意のBGM1本を選択してください。選択後、HTML画面が表示されます。',flush=True)
    uploaded=files.upload()
    names=[name for name in uploaded if Path(name).suffix.lower() in {'.mp3','.wav','.m4a','.ogg','.flac'}]
    if not 1<=len(names)<=2:raise ValueError('元音声1本と、必要ならBGM1本を選択してください')
    source_name=names[0];bgm_name=None
    if len(names)==2:
        for index,name in enumerate(names,1):print(f'{index}: {name}')
        choice=input('元のラジオ音声の番号（1か2）: ').strip()
        if choice not in ('1','2'):raise ValueError('1か2を入力してください')
        source_name=names[int(choice)-1];bgm_name=names[2-int(choice)]
    directory=Path(tempfile.mkdtemp(prefix='campus-source-'));audio=directory/Path(source_name).name
    audio.write_bytes(uploaded[source_name])
    bgm=None
    if bgm_name:
        bgm=directory/Path(bgm_name).name;bgm.write_bytes(uploaded[bgm_name])
    del uploaded
    bridge=HtmlBridge(audio,bgm)
    callback='campus_'+uuid.uuid4().hex
    output.register_callback(callback,lambda action,data:JSON(bridge.call(action,data)))
    display(HTML(page.replace('__CAMPUS_CALLBACK__',callback)))
    print('この下のHTML画面で操作してください。切断後は起動し直しと元音声の選択が必要です。',flush=True)
    return bridge

LIVE_HTML = '<!doctype html><html lang="ja"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Campus Voice | 切り抜き編集</title><style>:root{font-family:system-ui,sans-serif;color:#22342f;background:#f5f4ef}body{margin:0}header{background:#173e35;color:white;padding:24px 6%;display:flex;gap:24px;align-items:center}header span{font-weight:800;letter-spacing:3px}main{max-width:920px;margin:48px auto;padding:0 20px}h1{font-size:36px}h2{font-size:20px}section{background:white;border:1px solid #dedfd6;border-radius:16px;padding:28px;margin:24px 0}label{display:block;margin:16px 0;font-size:14px}input,select,textarea{display:block;box-sizing:border-box;width:100%;padding:10px;border:1px solid #bac8bf;border-radius:6px;margin-top:8px;font:inherit}button{background:#23604e;color:white;border:0;border-radius:8px;padding:12px 18px;margin:6px 8px 6px 0;cursor:pointer}button:hover{background:#173e35}audio{width:100%;margin-top:12px}.row{display:flex;gap:20px}.row label{flex:1}.candidate{border-top:1px solid #ddd;padding:14px 0}.segment{border-top:1px solid #eee;padding:10px 0}.segment input{max-width:240px}p,small{line-height:1.7;color:#62746b}header small{color:#c6d9ce}\n\nmain{max-width:1200px}#editor{display:grid;grid-template-columns:minmax(260px,1fr) minmax(320px,1.25fr);gap:24px;align-items:start}#editor[hidden]{display:none}#editor>section{margin:0}#editor>section:last-child{grid-column:1/-1}.candidate{padding:20px;background:#f5f8f5;border:1px solid #d9e4dc;border-radius:12px;margin:12px 0}.candidate h3{margin:0 0 8px;font-size:17px}.candidate p{font-size:13px}#candidates{max-height:75vh;overflow:auto}textarea{min-height:90px;resize:vertical}.secondary{background:#e6eee9;color:#173e35}.secondary:hover{background:#cfdfd5}#duration{font-weight:700;color:#173e35}#saveState{font-weight:700}button:focus-visible,input:focus-visible,select:focus-visible,textarea:focus-visible{outline:3px solid #bd791c;outline-offset:3px}@media(max-width:760px){#editor{display:block}#editor>section{margin:20px 0}.row{flex-wrap:wrap}.row label{min-width:120px}header{flex-wrap:wrap}section{padding:18px}#candidates{max-height:50vh}}\n\n.candidate-summary{color:#22342f!important;line-height:1.8}.candidate details{margin:14px 0;font-size:13px}.candidate summary{cursor:pointer;font-weight:600}.candidate li{margin:8px 0;line-height:1.6}\n</style><body><header><span>CAMPUS VOICE</span><small>ラジオのひと場面を、動画に。</small></header><main><section><h2>Colabとの一時接続</h2><p>この画面はColab内で動作します。公開サーバーは使用しません。切断前に作業全体を保存してください。</p>\n<label>解析範囲<select id="liveSeconds"><option value="0">全編</option><option value="180">冒頭3分だけ</option></select></label>\n<label>出演人数<input id="liveSpeakers" type="number" min="1" max="10" value="2"></label>\n<label>候補数<input id="liveCount" type="number" min="5" max="10" value="10"></label>\n<label><input id="liveDiarize" type="checkbox" checked>話者を推定</label><label><input id="liveTopics" type="checkbox" checked>話題の区切りも使う</label>\n<label><input id="liveBgm" type="checkbox">最初に選択したBGMを動画に使う</label><button id="liveAnalyze" data-live>解析する</button><pre id="liveStatus" role="status">接続を確認しています…</pre></section><h1>切り抜きをつくる</h1><p>Colabで解析 → ここで選んで編集 → Colabで動画を書き出し</p><section><h2>1. 素材を読み込む</h2><label>解析結果・保存した作業（JSON）<input id="project" type="file" accept=".json"></label><label>試聴音声はColabから取得します（音声の再選択は不要）<input hidden id="audioFile" type="file" accept="audio/*"></label><audio id="audio" controls></audio><p id="analysisInfo"></p><p id="status" role="status">解析結果を読み込んでください。</p></section><div id="editor" hidden><section><h2>2. 候補を選ぶ</h2><p>新しいColabでは話題の区切り・発話密度・テンポ・反応語で候補を提案します。反応語は笑い声の検出ではありません。話者はColabで推定します。誤りは試聴して修正してください。自動提案は確認用の目安です。前後を聴いて内容を判断してください。</p><div class="row"><label>候補・字幕を検索<input id="search" type="search" placeholder="気になる言葉を入力"></label><label>並べ替え<select id="sort"><option value="score">おすすめ順</option><option value="time">時刻順</option></select></label></div><p id="candidateCount"></p><div id="candidates"></div></section><section><h2>3. 内容を整える</h2><label>タイトル<input id="title"></label><div class="row"><label>開始（秒）<input id="start" type="number" min="0" step="0.1"></label><label>終了（秒）<input id="end" type="number" min="0" step="0.1"></label></div><p id="duration" role="status"></p><button id="useStart" class="secondary">再生位置を開始にする</button><button id="useEnd" class="secondary">再生位置を終了にする</button><br><button id="listen">選択範囲を試聴</button><button id="context">開始の10秒前から聴く</button><p>字幕は直接修正できます。解析時の話者名がある場合は、各欄で名前も変更できます。</p><details><summary>話者の名前をまとめて設定</summary><div id="speakers"></div></details><div id="segments"></div></section><section><h2>4. 書き出し設定</h2><label>動画サイズ<select id="format"><option value="vertical">縦型 9:16（Shorts / Reels）</option><option value="square">正方形 1:1</option><option value="landscape">横型 16:9</option></select></label><label>BGM音量（元音声に対する倍率）<input id="volume" type="number" min="0" max="1" step="0.01" value="0.12"></label><p>BGMファイルは動画生成時にColabで選びます。利用できる権利のある音源をご用意ください。</p><p id="saveState" role="status">読み込み済み</p><button id="saveSession">作業全体を保存（再編集用）</button><button id="copyEdit">Colabへコピー（ファイル送信不要）</button><label id="copyFallback" hidden>コピーできない場合は、下の内容をすべて選択してコピーしてください<textarea id="copyText" readonly></textarea></label><button id="liveRender" data-live>この内容で動画を作る</button><div id="liveResult"></div><p>BGMはColab起動時に選択したものを使えます。</p><button id="download">Colab用の編集JSONを保存</button><p>「Colabへコピー」→Colabの入力欄に貼り付け→動画生成。元音声とBGMは最初にColabへ選んだものを使います。</p></section></div><button id="clear">読み込みデータを消去</button></main><script type="module">function validateProject(p) {\n  if (typeof p?.audio_name !== \'string\' || !p.audio_name || !p.segments?.length) throw Error(\'元音声名と字幕が必要です\');\n  if (p?.version !== 1 || !Array.isArray(p.segments) || !Array.isArray(p.candidates)) throw Error(\'対応する解析結果ではありません\');\n  for (const s of p.segments) if (!Number.isFinite(s.start) || !Number.isFinite(s.end) || s.start < 0 || s.end <= s.start || typeof s.text !== \'string\') throw Error(\'字幕の時刻・内容が不正です\');\n  for (const c of p.candidates) if (!Number.isFinite(c.start) || !Number.isFinite(c.end) || c.start < 0 || c.end <= c.start) throw Error(\'候補の時刻が不正です\');\n  return p;\n}\nfunction exportEdit(project, start, end, segments, bgmVolume, format) {\n  if (![\'vertical\',\'square\',\'landscape\'].includes(format)) throw Error(\'動画サイズが不正です\');\n  if (!Number.isFinite(bgmVolume) || bgmVolume < 0 || bgmVolume > 1) throw Error(\'BGM音量は0〜1にしてください\');\n  if (!Number.isFinite(start) || !Number.isFinite(end) || start < 0 || end <= start) throw Error(\'終了時刻は開始時刻より後にしてください\');\n  if (!segments.some(s => s.end > start && s.start < end)) throw Error(\'字幕のある範囲を選んでください\');\n  return {version:1, audio_name:project.audio_name, start,end,segments:segments.filter(s=>s.end>start&&s.start<end),bgm_volume:bgmVolume,format};\n}\n\n\nconst $=id=>document.getElementById(id);\nlet project,url,stopAt,dirty=false;\nfunction status(message){$(\'status\').textContent=message}\nfunction changed(){dirty=true;$(\'saveState\').textContent=\'未保存の変更があります\'}\nfunction range(start,end){$(\'start\').value=start;$(\'end\').value=end;renderSegments();updateDuration()}\nfunction renderSegments(){\n  const root=$(\'segments\');root.replaceChildren();\n  for(const s of project.segments){\n    if(s.end<=+$(\'start\').value||s.start>=+$(\'end\').value)continue;\n    const box=document.createElement(\'div\');box.className=\'segment\';\n    const time=document.createElement(\'small\');time.textContent=`${s.start.toFixed(1)}–${s.end.toFixed(1)}秒`;\n    const speaker=document.createElement(\'input\');speaker.placeholder=\'話者名（任意）\';speaker.setAttribute(\'aria-label\',\'話者名\');speaker.value=s.speaker||\'\';\n    speaker.oninput=()=>{s.speaker=speaker.value;changed()};\n    const text=document.createElement(\'textarea\');text.setAttribute(\'aria-label\',`${s.start.toFixed(1)}秒の字幕`);text.value=s.text;\n    text.oninput=()=>{s.text=text.value;changed()};box.append(time,speaker,text);root.append(box);\n  }\n}\nfunction loadProject(p,selection){\n  project=validateProject(p);$(\'editor\').hidden=false;\n  renderCandidates();renderSpeakers();\n  const c=selection||project.candidates[0]||{start:project.segments[0].start,end:Math.min(project.segments.at(-1).end,project.segments[0].start+60)};\n  range(c.start,c.end);$(\'title\').value=c.title||\'\';$(\'volume\').value=c.bgm_volume??0.12;$(\'format\').value=c.format||\'vertical\';\n  dirty=false;$(\'saveState\').textContent=\'読み込み済み\';\n  $(\'analysisInfo\').textContent=project.analysis?.test_seconds ? `冒頭${project.analysis.test_seconds}秒の解析です。全編ではありません。` : (project.analysis?.test_seconds===0 ? \'全編を解析したデータです。\' : \'解析範囲は不明です。\');\n}\n$(\'project\').onchange=async e=>{\n  const file=e.target.files[0];if(!file)return;\n  if(dirty&&!confirm(\'未保存の編集があります。新しいデータを読み込みますか？\'))return;\n  try{\n    const data=JSON.parse(await file.text());\n    if(data.kind===\'campus-voice-session\'){\n      validateProject(data.project);\n      exportEdit(data.project,data.selection.start,data.selection.end,data.project.segments,data.selection.bgm_volume,data.selection.format);\n      loadProject(data.project,data.selection);\n    }else loadProject(data);\n    status(\'読み込みました。試聴するには元音声を選択してください。\');\n  }catch(e){status(e.message)}\n};\n$(\'audioFile\').onchange=e=>{\n  const file=e.target.files[0];if(!file)return;\n  if(project&&file.name!==project.audio_name){status(`元音声は「${project.audio_name}」を選んでください。`);e.target.value=\'\';return}\n  $(\'audio\').pause();stopAt=null;if(url)URL.revokeObjectURL(url);\n  url=URL.createObjectURL(file);$(\'audio\').src=url;status(\'音声を読み込みました。候補の前後を確認してください。\');\n};\nfor(const id of [\'start\',\'end\'])$(id).onchange=()=>{renderSegments();updateDuration();changed()};\nfor(const id of [\'title\',\'volume\',\'format\'])$(id).oninput=changed;\nasync function listen(context=false){\n  try{\n    const start=+$(\'start\').value,end=+$(\'end\').value;\n    exportEdit(project,start,end,project.segments,+$(\'volume\').value,$(\'format\').value);\n    if(!$(\'audio\').src||!Number.isFinite($(\'audio\').duration))throw Error(\'元音声を読み込み、準備ができるまでお待ちください\');\n    if(end>$(\'audio\').duration+0.1)throw Error(\'選択範囲が音声の長さを超えています\');\n    stopAt=Math.min($(\'audio\').duration,end+(context?10:0));$(\'audio\').currentTime=Math.max(0,start-(context?10:0));\n    await $(\'audio\').play();\n  }catch(e){status(e.message)}\n}\n$(\'listen\').onclick=()=>listen();$(\'context\').onclick=()=>listen(true);\n$(\'audio\').ontimeupdate=()=>{if(stopAt&&$(\'audio\').currentTime>=stopAt){$(\'audio\').pause();stopAt=null}};\nfunction selection(){const edit=exportEdit(project,+$(\'start\').value,+$(\'end\').value,project.segments,+$(\'volume\').value,$(\'format\').value);edit.title=$(\'title\').value;return edit}\nfunction download(data,name){const u=URL.createObjectURL(new Blob([JSON.stringify(data,null,2)],{type:\'application/json\'}));const a=document.createElement(\'a\');a.href=u;a.download=name;a.click();setTimeout(()=>URL.revokeObjectURL(u),1000)}\n$(\'download\').onclick=()=>{try{download(selection(),\'clip-edit.json\');status(\'Colab用の編集JSONを保存しました。再編集用には「作業全体を保存」も使ってください。\')}catch(e){status(e.message)}};\n$(\'saveSession\').onclick=()=>{try{download({kind:\'campus-voice-session\',version:1,project,selection:selection()},\'campus-voice-session.json\');dirty=false;$(\'saveState\').textContent=\'作業を保存しました\';status(\'作業全体を保存しました。次回はこのJSONと元音声を読み込むと再開できます。\')}catch(e){status(e.message)}};\n$(\'clear\').onclick=()=>{\n  if(dirty&&!confirm(\'未保存の編集を破棄して消去しますか？\'))return;\n  $(\'copyText\').value=\'\';$(\'copyFallback\').hidden=true;\n  project=null;dirty=false;stopAt=null;$(\'editor\').hidden=true;$(\'audio\').pause();$(\'audio\').removeAttribute(\'src\');$(\'audio\').load();\n  if(url)URL.revokeObjectURL(url);url=null;$(\'project\').value=\'\';$(\'audioFile\').value=\'\';$(\'candidates\').replaceChildren();$(\'segments\').replaceChildren();\n  status(\'読み込んだデータを消去しました。端末に保存済みのファイルは別途削除してください。\');\n};\nwindow.addEventListener(\'beforeunload\',e=>{if(dirty){e.preventDefault();e.returnValue=\'\'}});\n\nfunction updateDuration(){\n  const duration=+$(\'end\').value-+$(\'start\').value;\n  $(\'duration\').textContent=Number.isFinite(duration)&&duration>0 ? `${duration.toFixed(1)}秒${duration<30||duration>60?\' · おすすめは30〜60秒です\':\'\'}` : \'開始・終了を確認してください\';\n}\nfunction renderCandidates(){\n  const root=$(\'candidates\');root.replaceChildren();\n  if(!project)return;\n  const query=$(\'search\').value.trim().toLowerCase();\n  const candidates=project.candidates.filter(c=>`${c.title||\'\'} ${c.summary||\'\'} ${c.reason||\'\'} ${project.segments.filter(s=>s.end>c.start&&s.start<c.end).map(s=>s.text).join(\' \')}`.toLowerCase().includes(query));\n  candidates.sort($(\'sort\').value===\'time\' ? (a,b)=>a.start-b.start : (a,b)=>(b.score||0)-(a.score||0));\n  $(\'candidateCount\').textContent=`${candidates.length}件の候補`;\n  if(!candidates.length){const empty=document.createElement(\'p\');empty.textContent=\'候補がありません。検索を変えるか、開始・終了を手動で指定してください。\';root.append(empty)}\n  for(const [i,c] of candidates.entries()){\n    const card=document.createElement(\'article\');card.className=\'candidate\';\n    const heading=document.createElement(\'h3\');heading.textContent=`${i+1}. ${c.title||\'候補\'}`;\n    const detail=document.createElement(\'small\');detail.textContent=`${clock(c.start)}〜${clock(c.end)} · ${(c.end-c.start).toFixed(1)}秒`;\n    const summary=document.createElement(\'p\');summary.className=\'candidate-summary\';\n    summary.textContent=c.summary ? `内容の要約${c.summary_method===\'extractive-v1\'?\'（字幕から重要文を抜粋）\':\'\'}：${c.summary}` : \'内容の要約：この解析データにはありません。新版Colabで候補を再計算すると追加されます。\';\n    const reason=document.createElement(\'details\');\n    const label=document.createElement(\'summary\');label.textContent=\'この候補が選ばれた理由\';reason.append(label);\n    const list=document.createElement(\'ul\');\n    for(const text of (Array.isArray(c.reasons)&&c.reasons.length?c.reasons:[c.reason||\'理由の情報がありません。\'])){\n      const item=document.createElement(\'li\');item.textContent=text;list.append(item);\n    }\n    reason.append(list);\n    const select=()=>{range(c.start,c.end);$(\'title\').value=c.title||\'\';changed()};\n    const edit=document.createElement(\'button\');edit.textContent=\'この範囲を編集\';edit.onclick=()=>{select();$(\'title\').focus()};\n    const preview=document.createElement(\'button\');preview.className=\'secondary\';preview.textContent=\'前後を試聴\';preview.onclick=()=>{select();listen(true)};\n    card.append(heading,detail,summary,reason,edit,preview);root.append(card);\n  }\n}\nfunction clock(value){return `${Math.floor(value/60)}:${Math.floor(value%60).toString().padStart(2,\'0\')}`}\n$(\'search\').oninput=renderCandidates;$(\'sort\').onchange=renderCandidates;\n$(\'useStart\').onclick=()=>{if(Number.isFinite($(\'audio\').duration)){range(Number($(\'audio\').currentTime.toFixed(1)),+$(\'end\').value);changed()}else status(\'元音声を先に選択してください\')};\n$(\'useEnd\').onclick=()=>{if(Number.isFinite($(\'audio\').duration)){range(+$(\'start\').value,Number($(\'audio\').currentTime.toFixed(1)));changed()}else status(\'元音声を先に選択してください\')};\n\nfunction renderSpeakers(){\n  const root=$(\'speakers\');root.replaceChildren();\n  const speakers=[...new Set(project.segments.map(s=>s.speaker).filter(Boolean))];\n  if(!speakers.length){root.textContent=\'話者は未推定です。字幕ごとの話者欄で入力できます。\';return}\n  for(const name of speakers){\n    const label=document.createElement(\'label\');label.textContent=`${name} の名前`;\n    const input=document.createElement(\'input\');input.value=name;input.setAttribute(\'aria-label\',`${name}の名前`);\n    const button=document.createElement(\'button\');button.textContent=\'この話者をまとめて変更\';button.className=\'secondary\';\n    button.onclick=()=>{const next=input.value.trim();if(!next){status(\'名前を入力してください\');return}for(const s of project.segments){if(s.speaker===name)s.speaker=next}changed();renderSegments();renderSpeakers()};\n    label.append(input,button);root.append(label);\n  }\n}\n\n$(\'copyEdit\').onclick=async()=>{\n  try{\n    const text=JSON.stringify(selection());\n    $(\'copyText\').value=text;\n    try{\n      await navigator.clipboard.writeText(text);\n      $(\'copyFallback\').hidden=true;\n      status(\'コピーしました。Colabの編集データ読み込みセルを実行し、入力欄へ貼り付けてEnterを押してください。\');\n    }catch{\n      $(\'copyFallback\').hidden=false;$(\'copyText\').focus();$(\'copyText\').select();\n      status(\'下の選択済みテキストをCtrl+C（Macは⌘C）でコピーし、Colabへ貼り付けてください。\');\n    }\n  }catch(e){status(e.message)}\n};\n\nconst embeddedProject=document.getElementById(\'embeddedProject\');\nif(embeddedProject){\n  try{loadProject(JSON.parse(embeddedProject.textContent));status(\'解析結果は読み込み済みです。試聴用の元音声を選択してください。\')}\n  catch(e){status(`解析結果を読み込めませんでした: ${e.message}`)}\n}\n\n// This extension runs only in a Colab HTML output, never via an exposed server.\nconst callbackName=\'__CAMPUS_CALLBACK__\';\nconst liveStatus=$(\'liveStatus\');\nlet liveBusy=false,previewOffset=0;\nasync function rpc(action,data={}){\n  if(!globalThis.google?.colab?.kernel?.invokeFunction)throw Error(\'Colabへの接続がありません。このHTMLはColabの出力欄で操作してください。切断時は作業JSONを保存してから再起動します。\');\n  const reply=await google.colab.kernel.invokeFunction(callbackName,[action,data],{});\n  const result=reply.data[\'application/json\'];\n  if(!result?.ok)throw Error(result?.error||\'Colabが応答しませんでした\');\n  return result;\n}\nasync function runJob(action,data){\n  if(liveBusy)throw Error(\'前の処理が終わるまでお待ちください\');\n  liveBusy=true;\n  document.querySelectorAll(\'[data-live]\').forEach(b=>b.disabled=true);\n  try{\n    const {job}=await rpc(action,data);\n    for(;;){\n      const state=await rpc(\'status\',{job});\n      liveStatus.textContent=state.messages?.join(\'\\n\')||\'処理しています…\';\n      if(state.done)return {job,result:state.result,blob:state.result?.mime?await receive(job,state.result):null};\n      await new Promise(resolve=>setTimeout(resolve,2000));\n    }\n  }finally{liveBusy=false;document.querySelectorAll(\'[data-live]\').forEach(b=>b.disabled=false)}\n}\nasync function receive(job,result){\n  const chunks=[];let offset=0;\n  for(;;){\n    const part=await rpc(\'chunk\',{job,offset});\n    chunks.push(Uint8Array.from(atob(part.data),c=>c.charCodeAt(0)));offset=part.next;\n    if(part.done)break;\n  }\n  return new Blob(chunks,{type:result.mime});\n}\nfunction liveError(error){liveStatus.textContent=error.message;status(error.message)}\n$(\'liveAnalyze\').onclick=async()=>{\n  if(dirty&&!confirm(\'未保存の編集があります。作業を保存してから解析し直してください。続けますか？\'))return;\n  try{\n    const {result}=await runJob(\'analyze\',{seconds:+$(\'liveSeconds\').value,speakers:+$(\'liveSpeakers\').value,count:+$(\'liveCount\').value,diarize:$(\'liveDiarize\').checked,topics:$(\'liveTopics\').checked});\n    loadProject(result.project);liveStatus.textContent=\'解析完了。候補を選び、前後を試聴して編集してください。\';\n  }catch(e){liveError(e)}\n};\nlisten=async(context=false)=>{\n  try{\n    const edit=selection();const start=Math.max(0,edit.start-(context?10:0));\n    const {blob}=await runJob(\'preview\',{start,end:edit.end+(context?10:0)});\n    if(url)URL.revokeObjectURL(url);url=URL.createObjectURL(blob);\n    $(\'audio\').src=url;previewOffset=start;stopAt=null;await $(\'audio\').play();liveStatus.textContent=\'試聴音声を再生しています。\';\n  }catch(e){liveError(e)}\n};\n$(\'useStart\').onclick=()=>{if(Number.isFinite($(\'audio\').duration)){range(previewOffset+$(\'audio\').currentTime,+$(\'end\').value);changed()}};\n$(\'useEnd\').onclick=()=>{if(Number.isFinite($(\'audio\').duration)){range(+$(\'start\').value,previewOffset+$(\'audio\').currentTime);changed()}};\n$(\'liveRender\').onclick=async()=>{\n  try{\n    const edit=selection();const {blob}=await runJob(\'render\',{edit,bgm:$(\'liveBgm\').checked});\n    // Download immediately: later preview/analysis replaces the backend\'s last job.\n    const link=document.createElement(\'a\');\n    const u=URL.createObjectURL(blob);link.href=u;link.download=\'clip.mp4\';link.textContent=\'完成MP4を保存\';\n    link.onclick=()=>setTimeout(()=>URL.revokeObjectURL(u),60000);\n    $(\'liveResult\').replaceChildren(link);liveStatus.textContent=\'動画が完成しました。「完成MP4を保存」を押してください。\';\n  }catch(e){liveError(e)}\n};\nrpc(\'hello\').then(data=>{$(\'liveBgm\').disabled=!data.bgm;$(\'liveBgm\').checked=data.bgm;liveStatus.textContent=`接続しました。元音声：${data.audio_name}。「解析する」を押してください。`}).catch(liveError);\n</script></body></html>\n'
campus_bridge = launch_html_editor(LIVE_HTML)
